In [ ]:
#The model has four hyperparameters to tune, and its output reflects the differnece
#from the expensive baseline. Because the system is dynamic and full of local optima, 
#it requires careful tuning and robust validation to find reliable, near-optimal solutions

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from IPython.display import clear_output
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
from scipy.interpolate import griddata

# Your 4D spatial coordinate data (10 positions)
inputs = np.load('/home/mcevoyj777/_CAPSTONE/FUNC4/initial_inputs.npy')

# Your ouput values
output = np.load('/home/mcevoyj777/_CAPSTONE/FUNC4/initial_outputs.npy')
print(output.max())
extra_inputs = [[0.413038, 0.408189, 0.347264, 0.431326],[0.413038, 0.408189, 0.347264, 0.431326],
                [0.435949, 0.443948, 0.243355, 0.439113],[0.344064, 0.436389, 0.466159, 0.458097],
               [0.473737, 0.329747, 0.407231, 0.480575],[0.463369, 0.565191, 0.411343, 0.429494],
               	[0.484691, 0.419585, 0.363640, 0.389904],	[0.414488, 0.445090, 0.366821, 0.528614]]
extra_output = [0.5481902259003815,0.5481902259003815,-1.847789200471531,
                -0.9368462707140419,-2.1240106008010033,-3.291222601614592,
               -1.0738604820061286,-1.6546686947288474]

inputs = np.vstack([inputs, extra_inputs])
output = np.append(output, extra_output)
print(output.max())
print(inputs)
print(output)

# 2. Combine into a single Pandas DataFrame
df = pd.DataFrame(inputs, columns=['Input_W','Input_X', 'Input_Y', 'Input_Z'])
df['Output_Target'] = output

# 3. Compute the correlation matrix
# Change method to 'spearman' if relationships are non-linear curves
corr_matrix = df.corr(method='pearson')

# 4. Isolate how each input correlates *only* with the target output
target_corr = corr_matrix['Output_Target'].drop('Output_Target')

print("Correlation with Output:")
print(target_corr)

# Calculate non-linear relationship dependency (higher score = stronger relationship)
mi_scores = mutual_info_regression(inputs, output)

for name, score in zip(['Input_W','Input_X', 'Input_Y', 'Input_Z'], mi_scores):
    print(f"{name} Mutual Information Score: {score:.4f}")

# Fit a quick non-linear model that checks all 4 dimensions simultaneously
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(inputs, output)

# Extract total feature importances (including interactions)
importances = model.feature_importances_

for name, score in zip(['Input_W','Input_X', 'Input_Y', 'Input_Z'], importances):
    print(f"{name} Total Importance (with interactions): {score:.4f}")

-4.025542281908162
0.5481902259003815
[[0.89698105 0.72562797 0.17540431 0.70169437]
 [0.8893564  0.49958786 0.53926886 0.50878344]
 [0.25094624 0.03369313 0.14538002 0.49493242]
 [0.34696206 0.0062504  0.76056361 0.61302356]
 [0.12487118 0.12977019 0.38440048 0.2870761 ]
 [0.80130271 0.50023109 0.70664456 0.19510284]
 [0.24770826 0.06044543 0.04218635 0.44132425]
 [0.74670224 0.7570915  0.36935306 0.20656628]
 [0.40066503 0.07257425 0.88676825 0.24384229]
 [0.6260706  0.58675126 0.43880578 0.77885769]
 [0.95713529 0.59764438 0.76611385 0.77620991]
 [0.73281243 0.14524998 0.47681272 0.13336573]
 [0.65511548 0.07239183 0.68715175 0.08151656]
 [0.21973443 0.83203134 0.48286416 0.08256923]
 [0.48859419 0.2119651  0.93917791 0.37619173]
 [0.16713049 0.87655456 0.21723954 0.95980098]
 [0.21691119 0.16608583 0.24137226 0.77006248]
 [0.38748784 0.80453226 0.75179548 0.72382744]
 [0.98562189 0.66693268 0.15678328 0.8565348 ]
 [0.03782483 0.66485335 0.16198218 0.25392378]
 [0.68348638 0.9027701

In [2]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# Train a predictive model on your 20 existing samples
predictor = RandomForestRegressor(n_estimators=100, random_state=42)
predictor.fit(inputs, output)

def objective_4d(X):
    """
    Surrogate function for 4D optimization.
    X: A 2D numpy array of shape (n_samples, 4) sent by the optimizer.
    """
    # Force input to be 2D array matrix for sklearn compatibility
    X = np.atleast_2d(X)
    
    # Predict the output value using the trained model
    predictions = predictor.predict(X)
    
    # Return as a flat 1D array as expected by your code (.ravel())
    return predictions.ravel()

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, Matern
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

def upper_confidence_bound(mu, sigma, kappa=0.1):
    """
    Upper Confidence Bound (UCB) acquisition function.
    
    UCB = mean + kappa * std
    
    Parameters:
    -----------
    mu : predicted mean
    sigma : predicted standard deviation
    kappa : exploration parameter (higher = more exploration)
    """
    return mu + kappa * sigma


def expected_improvement(mu, sigma, y_best, xi=0.05):
    """
    Expected Improvement (EI) acquisition function.
    
    EI = E[max(f(x) - f(x_best), 0)]
    
    Parameters:
    -----------
    mu : predicted mean
    sigma : predicted standard deviation  
    y_best : best observed value so far
    xi : exploration parameter
    """
    with np.errstate(divide='warn'):
        improvement = mu - y_best - xi
        Z = improvement / sigma
        ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei


In [7]:
def bayesian_optimization_nd(objective_func, n_dims, n_iterations=1, 
                            n_initial=1, acquisition='ei'):
    """
    Bayesian Optimization for n-dimensional functions.
    Assumes bounds [0, 1] for all dimensions.
    """
    bounds = [(0, 1) for _ in range(n_dims)]
    
    # Initialize
    X_samples = inputs
    y_samples = output
    
    
    best_values = [y_samples.max()]
    
    print(f"Starting {n_dims}D optimization with {n_initial} initial samples...")
    print(f"Initial best: {y_samples.max():.4f}\n")
  
    
    for iteration in range(n_iterations):
        # Fit GP
        kernel = ConstantKernel(1.0) * Matern(length_scale=0.3,nu=2.5)
        gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,normalize_y=True,
                                     n_restarts_optimizer=5)
        gp.fit(X_samples, y_samples)

        # Optimize acquisition function
        def acq_objective(X):
            X = X.reshape(1, -1)
            mu, sigma = gp.predict(X, return_std=True)
            
            if acquisition == 'ei':
                acq_val = expected_improvement(mu, sigma, y_samples.max())
            else:
                acq_val = upper_confidence_bound(mu, sigma, kappa=1)
           
            return -acq_val
        
        # Multi-start optimization
        best_acq = np.inf
        X_next = None
        # Build dynamic, localized acquisition optimization bounds
        #
        localized_bounds = [
            (0.35, 0.5),  # Input_W restricted search
            (0.35, 0.5),  # Input_X restricted search
            (0.2, 0.45),  # Input_Y restricted search
            (0.35, 0.55),  # Input_Z restricted search
        ]

        
        for _ in range(50):  # More starts for higher dimensions
            # Sample initial multi-start points respecting the localized constraints
            x0 = np.array(
                [
                    np.random.uniform(localized_bounds[0][0], bounds[0][1]),
                    np.random.uniform(localized_bounds[1][0], bounds[1][1]),
                    np.random.uniform(localized_bounds[2][0], bounds[2][1]),
                    np.random.uniform(localized_bounds[3][0], bounds[3][1])
                ]
            )
            result = minimize(acq_objective, x0, bounds=localized_bounds, method='L-BFGS-B',options={"ftol": 1e-9},) 
      
            if result.fun < best_acq:
                best_acq = result.fun
                X_next = result.x
                print(f"Optimized Next Point Found! Predicted Acquisition Score: {-best_acq:.6f}")
                print(f"Coordinates to test next: {X_next}")

       
        # Evaluate
        y_next = objective_func(X_next.reshape(1, -1)).ravel()
        
        # Update
        X_samples = np.vstack([X_samples, X_next])
        y_samples = np.append(y_samples, y_next)
        best_values.append(y_samples.max())
        
        if (iteration + 1) % 10 == 0:
            print(f"Iteration {iteration + 1}: Best f(x)={y_samples.max():.4f}")
    
    return X_samples, y_samples, best_values



# Run 6D optimization
X_samples_4d, y_samples_4d, best_values_4d = bayesian_optimization_nd(
    objective_4d, 
    n_dims=4,
    n_iterations=3,
    acquisition='ei'
)

Starting 4D optimization with 1 initial samples...
Initial best: 0.5482

Optimized Next Point Found! Predicted Acquisition Score: 0.000000
Coordinates to test next: [0.5       0.5       0.2439727 0.55     ]
Optimized Next Point Found! Predicted Acquisition Score: 0.000000
Coordinates to test next: [0.5        0.5        0.23798814 0.55      ]
Optimized Next Point Found! Predicted Acquisition Score: 0.000000
Coordinates to test next: [0.4519466  0.35258071 0.45       0.44544672]
Optimized Next Point Found! Predicted Acquisition Score: 0.892749
Coordinates to test next: [0.35       0.38471599 0.34530858 0.40042474]
Optimized Next Point Found! Predicted Acquisition Score: 0.892749
Coordinates to test next: [0.35       0.38471609 0.34530853 0.40042466]
Optimized Next Point Found! Predicted Acquisition Score: 0.000114
Coordinates to test next: [0.5  0.5  0.2  0.55]
Optimized Next Point Found! Predicted Acquisition Score: 0.118304
Coordinates to test next: [0.40479091 0.42707165 0.38397768 0